In [ ]:
from testflows.combinatorics import Covering
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn import svm
import random
from sklearn.neighbors import KNeighborsClassifier
from numpy import inf
import time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
import umap


In [ ]:
import opfython.math.general as g
import opfython.stream.parser as p
import opfython.stream.splitter as s
from opfython.models import SupervisedOPF
from opfython.stream import loader
from opfython.utils import logging

In [ ]:
df_olive = pd.read_csv(r'../data/data_temp/Olive_Oils_Quadrum.csv')
df_olive

Load data set

In [ ]:
df_cis = pd.read_csv('../data/cis_data.csv')
df_cacao = pd.read_csv(r'..\\data\\data_temp\\cacao.csv')
df_algarrobo = pd.read_csv(r'..\\data\\data_temp\\algarrobo.csv')
df_fruits_pures = pd.read_csv(r'../data/data_temp/MIR_Fruit_purees.csv')
df_fresh_meat = pd.read_csv(r'../data/data_temp/Fresh_meats.csv')
df_olive = pd.read_csv(r'../data/data_temp/Olive_Oils_Quadrum.csv')


df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

algarrobo_x = df_algarrobo.loc[:, 'R':'REDVI']
y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1])
X_algarrobo = (algarrobo_x-algarrobo_x.min())/(algarrobo_x.max()-algarrobo_x.min())

cis_x = df_cis[['X', 'Y', 'X10', 'Y10', 'X20', 'Y20', 'X30', 'Y30', 'X40', 'Y40']]
y_cis = df_cis[['Result']]


fruits_pures_x = df_fruits_pures.iloc[:,1:]
y_fruit_puree = df_fruits_pures.iloc[:,0:1].replace(to_replace=['Not Strawberry', 'Strawberry'], value=[0, 1])
X_fruit_puree = (fruits_pures_x-fruits_pures_x.min())/(fruits_pures_x.max()-fruits_pures_x.min())


unique_names_meat =  set(df_fresh_meat["meat"])
meat_x = df_fresh_meat.iloc[:,4:]
y_meat = df_fresh_meat.iloc[:,0:1].replace(to_replace=unique_names,value=range(0,len(unique_names)))
X_meat =  (meat_x-meat_x.min())/(meat_x.max()-meat_x.min())

unique_names_olive = set(df_olive["Provenance"])
olive_x = df_olive.iloc[:,3:]
y_olive = df_olive.iloc[:,2:3].replace(to_replace=unique_names_olive,value=range(0,len(unique_names_olive)))
X_olive =  (olive_x-olive_x.min())/(olive_x.max()-olive_x.min())

X_cis = (cis_x-cis_x.min())/(cis_x.max()-cis_x.min())

In [ ]:
def ICAFS(dataset_X, dataset_Y, strenght,max_iteartion,model, print_logs=False):

  max_it = 1 
  score_list = []
  featur_list = []
  iter_list = []
  
  v_variable = [0, 1]
  best_f1_score = float('-inf')
  max_iteartion_aux =  max_iteartion
  best_data_set = dataset_X.columns.values.copy()

  initial_test_training(score_list,iter_list,featur_list,dataset_X,dataset_Y,model)
  while max_iteartion_aux > 0:

      dict_parameters = {}
      partial_best_list = []
      partial_score = 0
      for colum_key in best_data_set:
          dict_parameters[colum_key] = v_variable
      random.shuffle(best_data_set)
      generate_covering_array = Covering(dict_parameters, strength=strenght)
      for test in generate_covering_array.array:
          list_attributes_to_consider = []

          check_for_all_cero = True
          for (test_key, test_value) in test.items():
              if test_value == 1:
                  check_for_all_cero = False
                  list_attributes_to_consider.append(test_key)

          if check_for_all_cero:
              continue  
          df_temp = dataset_X[list_attributes_to_consider]
          clf_new = train_model(model)
          score = cross_val_score(clf_new,df_temp.values, dataset_Y.values.ravel(), cv=5, scoring='f1_macro')
          final_score = score.mean()

          if final_score >= partial_score :
              partial_score = final_score
              partial_best_list = list_attributes_to_consider
          clf_new = None
      best_data_set = partial_best_list.copy()
      best_f1_score = partial_score
      
      if print_logs:
        print(f"best f1 score= {best_f1_score}, iteration:{max_it}, numbers features selected ={ len(best_data_set)},best features selected={', '.join(best_data_set)}" )

      iter_list.append(max_it)
      aux_data_score = best_f1_score
      score_list.append(aux_data_score)
      max_iteartion_aux = max_iteartion_aux-1
      max_it = max_it +1
      featur_list.append(len(best_data_set))
  return iter_list,featur_list,score_list

def initial_test_training(score_list, iter_list, feature_list ,X,y,model):
     X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(X.values, y.values.ravel(), test_size=0.20, random_state=42)
     clf_new = train_model(model)
     clf_new.fit(X_train_temp, y_train_temp)
     y_pred = clf_new.predict(X_test_temp)
     score_list.append(f1_score(y_test_temp, y_pred,average='macro'))
     feature_list.append(X.shape[1])
     iter_list.append(0)

def train_model(model_name):  
    m = eval(model_name)
    return m

In [ ]:
def  plot_results_for_covering_array(scores,feature,num_of_iterarion, path_to_save_image):
        color = 'tab:blue'
        res_scores = np.array(scores)
        res_features = np.array(feature)
        res_iter = np.array(num_of_iterarion)

        plt.figure(figsize=(11, 10))
        
        fig, ax1 = plt.subplots()
        barwidth = 0.4
        color = 'tab:red'
        ax1.set_xlabel('Iterations')
        ax1.set_ylabel('Number of features', color=color)
        #ax1.set_title("ICAFS Feature selection on the Cacao dataset")
        ax1.spines['top'].set_visible(False)
        ax1.bar(res_iter-0.2, res_features, color=color, width=barwidth)
        ax1.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax1.set_ylim(1,max(res_features)+3)
        ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
        
        for bar in ax1.patches:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height}', fontsize=10,
                    ha='center', va='bottom', rotation=90)
            
        ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
        color = 'tab:blue'
        ax2.set_ylabel('F1_score', color=color)
        ax2.bar(res_iter+0.2, res_scores, color=color, width=barwidth)
        ax2.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax2.set_ylim(min(res_scores)-0.001, max(res_scores)+0.001)

        fig.tight_layout()  # otherwise the right y-label is slightly clipped
       
        for bar in ax2.patches:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height:.2f}', fontsize=10,
                    ha='center', va='bottom', rotation=90)

        #plt.title('ICAFS Feature selection for Cacao Dataset with OPF', y=-0.20)
        plt.gca().set_frame_on(False)
        plt.savefig(path_to_save_image)

In [ ]:

from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler


def plot_original_vs_reduced_umap(df_reduced_dataset,df_complete, y_dataset_integer):
    umap_3d = umap.UMAP(n_components=3)
    
    X_reduced_standarized = StandardScaler().fit_transform(df_reduced_dataset)
    X_complete_standarized  = StandardScaler().fit_transform(df_complete)
    
    
    reduced_umap = umap_3d.fit_transform(X_reduced_standarized)
    complete_umap = umap_3d.fit_transform(X_complete_standarized)
  
   

    #plt.savefig(path_to_save_image)
    # plotting
    #ax.scatter(x, y, z, c=[sns.color_palette()[x] for x in y_dataset_integer.values.ravel()])
    #ax.set_title('3D Algarrobo dataset reduced with UMAP with CAFS KNN')
    #plt.savefig( r'.\\output_images\\umap_.cafs_knn.png')

    fig = plt.figure(figsize=plt.figaspect(0.5))

    
    ax = fig.add_subplot(1, 2, 2, projection='3d')
    ax.scatter(complete_umap[:,0], complete_umap[:,1], complete_umap[:,2], c=[sns.color_palette()[x] for x in y_dataset_integer.values.ravel()])
    
    ax = fig.add_subplot(1, 2, 1, projection='3d')
    ax.scatter(reduced_umap[:,0], reduced_umap[:,1], reduced_umap[:,2], c=[sns.color_palette()[x] for x in y_dataset_integer.values.ravel()])

    plt.show()

ICAFS KNN

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_cacao,y_cacao,2,10,'KNeighborsClassifier(n_neighbors=2)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'./output_images/cacao_knn_icafs.png')

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_algarrobo,y_algarrobo,2,10,'KNeighborsClassifier(n_neighbors=3)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\algarrobo_knn_icafs.png')

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_fruit_puree,y_fruit_puree,2,10,'KNeighborsClassifier(n_neighbors=3)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\fruit_puree_knn_icafs.png')

In [ ]:
plot_original_vs_reduced_umap(X_fruit_puree[['1404.985', '1443.585', '1474.465', '1516.925', '1609.565', '1679.044', '1771.684', '1802.564']],X_fruit_puree,y_fruit_puree)


In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_meat,y_meat,2,10,'KNeighborsClassifier(n_neighbors=3)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\meat_knn_icafs.png')

In [ ]:
plot_original_vs_reduced_umap(X_meat[['1582.289', '1744.372', '1867.864']], X_meat,y_meat)

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_olive,y_olive,2,10,'KNeighborsClassifier(n_neighbors=3)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\olive_icafs.png')

In [ ]:
plot_original_vs_reduced_umap(X_olive[['893.4375', '986.0545', '1034.292', '1126.911', '1250.404', '1373.896', '1497.388', '1620.88', '1744.372', '1867.864']],X_olive,y_olive)

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_cis,y_cis,3,10,'KNeighborsClassifier(n_neighbors=3)',True)
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cis_knn_icafs.png')

ICAFS OPF

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_algarrobo,y_algarrobo,2,10,'SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\algarrobo_opf_icafs.png')

5 CV Algarrobo OPF

In [ ]:
from sklearn.model_selection import KFold

scores = []
kf = KFold(n_splits=5,shuffle=True, random_state =42)
X = X_algarrobo[[ 'NGRDI', 'NDVI', 'RVI', 'DVI', 'EVI', 'REVI', 'NDRE', 'RERVI', 'REDVI']]

for i, (train_index, test_index) in enumerate(kf.split(X.values)):
   
   x_fold_train = X.values[train_index,:]
   y_fold_train = y_algarrobo.values[train_index]

   x_fold_test =  X.values[test_index,:]
   y_fold_test = y_algarrobo.values[test_index]
   
   clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
   clf.fit(x_fold_train, y_fold_train)
   y_pred_fold = clf.predict(x_fold_test)
   score = f1_score(y_fold_test, y_pred_fold,average='macro')
   scores.append(score)

res = np.asarray(scores, dtype=np.float32)
print(f"5-CV Algarrobo OPF mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterations ,feature,score  = ICAFS(X_cacao,y_cacao,2,10,'SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)',True )
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cacao_opf_icafs.png')

5 CV Caco OPF

In [ ]:
from sklearn.model_selection import KFold

scores = []
kf = KFold(n_splits=5, shuffle=True, random_state =42)
X = X_cacao[['1542', '1611', '1836', '1930', '2398']]

for i, (train_index, test_index) in enumerate(kf.split(X.values)):
   
   x_fold_train = X.values[train_index,:]
   y_fold_train = y_cacao.values[train_index]

   x_fold_test =  X.values[test_index,:]
   y_fold_test = y_cacao.values[test_index]
   clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
   clf.fit(x_fold_train, y_fold_train)
   y_pred_fold = clf.predict(x_fold_test)
   score = f1_score(y_fold_test, y_pred_fold,average='macro')
   scores.append(score)

res = np.asarray(scores, dtype=np.float32)
print(f"5-CV Cacao OPF mean:{res.mean()} , 5-CV Cacao STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_cis,y_cis,3,4,'SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'./output_images/cis_opf_icafs.png')

5 CV OPF CIS

In [ ]:
from sklearn.model_selection import KFold

scores = []
kf = KFold(n_splits=5, shuffle=True, random_state =42)
X = X_cis[['X', 'Y', 'Y40']]

for i, (train_index, test_index) in enumerate(kf.split(X.values)):
   
   x_fold_train = X.values[train_index,:]
   y_fold_train = y_cis.values[train_index]

   x_fold_test =  X.values[test_index,:]
   y_fold_test = y_cis.values[test_index]
   clf = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
   clf.fit(x_fold_train, y_fold_train)
   y_pred_fold = clf.predict(x_fold_test)
   score = f1_score(y_fold_test, y_pred_fold,average='macro')
   scores.append(score)

res = np.asarray(scores, dtype=np.float32)
print(f"5-CV Cacao OPF mean:{res.mean()} , 5-CV Cacao STD:{res.std()}" )

ICAFS SVC

In [ ]:
start_time = time.time()
iterations ,feature,score  = ICAFS(X_cacao,y_cacao,2,10,'svm.SVC()' ,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cacao_svc_icafs.png')

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_algarrobo,y_algarrobo,2,10,'svm.SVC()',True )
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\algarrobo_svc_icafs.png')

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_fruit_puree,y_fruit_puree,2,10,'svm.SVC()',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\fruit_puree_svc_icafs.png')

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_cis,y_cis,3,10,'svm.SVC()',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cis_knn_icafs.png')

5 Cross Validation ICAFS CIS SVC

In [ ]:
clf = svm.SVC()
X = X_cis[['X', 'Y', 'Y40']]
res = cross_val_score(clf,X.values, y_cis.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV CIS mean:{res.mean()} , 5-CV CIS STD:{res.std()}" )

ICAFS MLP

In [ ]:
start_time = time.time()
iterations ,feature,score  = ICAFS(X_cacao,y_cacao,2,10,'MLPClassifier(solver="sgd", max_iter=7000, shuffle=False)',True )
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cacao_mlp_icafs.png')

ICAFS MLP CACAO 5 CV

In [ ]:
clf = MLPClassifier(solver="sgd", max_iter=7000, shuffle=False)
X = X_cacao[['1737', '1761', '1810', '1814', '1822', '1826', '1857', '1873', '1877', '1902', '1949', '1953', '1961', '1965', '1989', '2006', '2010', '2038', '2082', '2086', '2097', '2101', '2126', '2173', '2185', '2285', '2401', '2406', '2437']
]
res = cross_val_score(clf,X.values, y_cacao.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV Cacao mean:{res.mean()} , 5-CV Cacao STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_algarrobo,y_algarrobo,2,10,'MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)' ,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\algarrobo_mlp_icafs.png')

ICAFS MLP Algarrobo 5 CV

In [ ]:
clf = MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)
X = X_algarrobo[['GBRI.1', 'CIVE', 'MGRVI']]
res = cross_val_score(clf,X.values, y_algarrobo.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV Algarrobo mean:{res.mean()} , 5-CV Algarrobo STD:{res.std()}" )

In [ ]:
start_time = time.time()
iterations ,feature,score = ICAFS(X_cis,y_cis,3,10,'MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)',True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\cis_mlp_icafs.png')

ICAFS MLP CIS 5 CV

In [ ]:
clf = MLPClassifier(solver="sgd", max_iter=5000, shuffle=False)
X = X_cis[['X', 'Y', 'Y40']]
res = cross_val_score(clf,X.values, y_cis.values.ravel(), cv=5,scoring = 'f1_macro')
print(f"5-CV CIS mean:{res.mean()} , 5-CV CIS STD:{res.std()}" )